# 24 · Reproject points, checked against pyproj

## What this checks

**Reproject points** turned longitude and latitude into UTM, a flat grid in metres for one zone of the Earth. This
check covers release 1.0.0, which projects in that direction: from longitude and latitude to UTM. Reproject points
is now retired.

In release 0.1.0 the widget worked the other way, from UTM back to longitude and latitude. [Check
01](01-reproject-vs-pyproj.ipynb) covers that direction. When the widget turned round, the only check it had covered
a job it no longer did. This check closes that gap.

A point projected wrongly is measured wrongly. So it is worth checking.

## How it is checked

We project the same points with **pyproj**, the Python link to PROJ. PROJ is the projection library most GIS software
uses. Fieldwork uses proj4 in the browser. The two share no code.

Projecting there and back with Fieldwork alone would prove nothing. Both directions come from the same proj4 build.
A fault in its UTM would come back to nine decimal places and still be wrong both ways. Only an independent
implementation can tell.

The data are 9 points in UTM zone 30 north.

## What should happen

1. Every point lands within 1 millimetre of where pyproj puts it.
2. The EPSG code Fieldwork reports names the same frame as the projection it actually used.

Why 1 millimetre? Fieldwork stores projected coordinates to three decimals. Rounding alone can move a point by up to
half a millimetre on each axis, about 0.7 mm in all. One millimetre is the smallest tolerance that rounding does not
already use up. It is also the tolerance check 01 uses for the other direction.

These criteria were fixed before the check was run.

In [ ]:
# pyproj ships in the Pyodide distribution rather than as a wheel on this site, so it
# loads with pyodide_js.loadPackage. Outside Pyodide this cell does nothing.
import sys

if "pyodide" in sys.modules:
    import pyodide_js
    await pyodide_js.loadPackage("pyproj")

In [ ]:
import json
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
from checks.check_24_project_forward import run, TOLERANCE_M

report = run(Path.cwd() / "fixtures" / "project-forward-utm30n-001.json")
print(json.dumps({k: report[k] for k in
                  ["environment", "fixture", "toleranceM", "points",
                   "maxSeparationM", "declaredCodeMatchesDefinition", "agrees"]}, indent=2))

## The result on record

First recorded on 14 October 2026, against release 0.2.0. Rerun on 26 November 2026, against release 1.0.0:
**Agrees**.

The worst point is 0.62 mm from pyproj's, over 9 points. The EPSG code names the same frame as the projection used.

The data are verified before anything is compared. If they do not match their recorded digest, the notebook stops.
A result computed on data nobody can identify is not evidence.

Running the cell above repeats the check on the same data. The cell below lists every point, worst first.

In [ ]:
for row in sorted(report["rows"], key=lambda r: -r["separationM"]):
    print(f"{row['id']:>4}  {row['input']['longitude']:>6.1f}, {row['input']['latitude']:>5.1f}"
          f"   reference {row['reference']['easting']:>12.3f}, {row['reference']['northing']:>12.3f}"
          f"   fieldwork {row['fieldwork']['easting']:>12.3f}, {row['fieldwork']['northing']:>12.3f}"
          f"   separation {row['separationM'] * 1000:6.3f} mm")

## Reading the residual

The differences are the size of the stored rounding, not of the projection.

Fieldwork rounds eastings and northings to three decimals. Under that rounding, two correct implementations can
differ by up to the square root of 0.0005² + 0.0005², about 0.00071 m. A difference at or below that bound means the
two agree, and the file cannot record the agreement more finely. A difference well above it would be a real
disagreement.

The points are spread on purpose:

- **The centre.** Some lie on the zone's central line, at −3°, where the easting is exactly 500,000 by design.
- **The edges.** Some lie at both edges of the zone, which is six degrees wide. That is where the map is most
  stretched.
- **The latitudes.** These run from 0.5° to 70°, because a metre north is a very different number of degrees at each.

The cell below computes the bound, so it is not just asserted.

In [ ]:
# The bound the residual is read against, computed rather than asserted.
import math

half_unit = 0.5 * 10 ** -report["fixture"]["coordinateDecimals"]
bound = math.hypot(half_unit, half_unit)
print(f"rounding bound          {bound * 1000:.3f} mm")
print(f"worst separation        {report['maxSeparationM'] * 1000:.3f} mm")
print(f"within rounding alone   {report['maxSeparationM'] <= bound}")
print(f"within stated tolerance {report['maxSeparationM'] <= TOLERANCE_M}")

## The check that the coordinates alone cannot make

Fieldwork builds two things from the same zone number and hemisphere:

- **The definition.** The proj4 definition it projects with.
- **The label.** The `EPSG:326xx` label its records carry.

They are built separately, so they can disagree. **No comparison of coordinates would notice.** Every point would
agree with the definition used. Yet the records would name a different frame to anyone reading them later.

So this check also asks PROJ whether the declared code is the same frame as the declared definition. On 14 October
2026 we changed the data on purpose, to see what the check would catch:

- **Nothing changed.** Separation 0.62 mm, code matches: agrees.
- **One easting moved 2 mm.** Separation 2.13 mm, code matches: disagrees.
- **The definition's zone changed from 30 to 31.** Separation 671,519 m, code does not match: disagrees.
- **The code changed from `EPSG:32630` to `EPSG:32631`.** Separation 0.62 mm, code does not match: disagrees.
- **The definition made southern.** Separation 10,000,000 m, code does not match: disagrees.

The fourth change is the one that matters. The coordinates are untouched and still agree to 0.62 mm. Only the label is
wrong. Without the frame check, this check would have passed a widget that misnames its projection in every record.

## What this does not show

- That zone 30 north is the right zone for any given data. The same points can be projected into any zone and still
  look valid.
- Anything about datums. Both sides are given WGS84. Data on another datum needs a shift that Fieldwork does not do,
  and this check does not test.
- That the widget is used correctly in a workflow. Every Fieldwork step that measures refuses projected points. That
  is a rule of the application, not a property of this projection.